In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 1.消息类型

在LangChain中，发送给LLM的消息、LLM返回的消息都统一被封装为BaseMessage，它是Agent中基本的上下文单元。

在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role）创建了对应的BaseMessage的子类：
- SystemMessage：role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage：role是user，代表用户输入的消息
- AIMessage：role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- ToolMessage：role是tool，代表工具调用时产生的结果

我们可以直接使用这些Messages类型来发送消息。

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 定义工具
@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"

# 创建Agent
agent = create_agent(model="deepseek-chat", tools=[get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        SystemMessage("请使用工具来获取天气信息。"),
        HumanMessage("你好，我是虎哥."),
        AIMessage("你好，虎哥，很高兴认识你."),
        HumanMessage("北京今天天气如何？")
    ]
})

print(response)

/Users/xiaor/Professional_accomplishment/中电智安/project/langchain_learn/langchain-learn/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'messages': [SystemMessage(content='请使用工具来获取天气信息。', additional_kwargs={}, response_metadata={}, id='67b6f29c-8ccc-4325-be7e-a55718519b0c'), HumanMessage(content='你好，我是虎哥.', additional_kwargs={}, response_metadata={}, id='0274cc60-989f-4369-bbdc-1c06a0007349'), AIMessage(content='你好，虎哥，很高兴认识你.', additional_kwargs={}, response_metadata={}, id='3e46f675-76df-40d4-b457-82fb856c03a1', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='北京今天天气如何？', additional_kwargs={}, response_metadata={}, id='cc7c4b09-e1b4-4a65-bbef-b98bb7e07789'), AIMessage(content='好的，虎哥，我来帮你查一下北京今天的天气！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 316, 'total_tokens': 372, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 316}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_pr

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [3]:
for message in response['messages']:
    message.pretty_print()

================================ System Message ================================

请使用工具来获取天气信息。
================================ Human Message =================================

你好，我是虎哥.
================================== Ai Message ==================================

你好，虎哥，很高兴认识你.
================================ Human Message =================================

北京今天天气如何？
================================== Ai Message ==================================

好的，虎哥，我来帮你查一下北京今天的天气！
Tool Calls:
  get_weather (call_00_SVOKvUZHviBdyS7HzbQB8458)
 Call ID: call_00_SVOKvUZHviBdyS7HzbQB8458
  Args:
    location: 北京
================================= Tool Message =================================
Name: get_weather

Current weather in 北京 is sunny
================================== Ai Message ==================================

北京今天天气晴朗（sunny）☀️，是个好天气！虎哥如果有出行计划的话，是个不错的日子哦。


# 2.多模态消息

之前我们都是向模型发送文本消息，但是 LangChain 也支持向模型发送多模态消息，比如图片、音频、视频、文本等。但前提是必须是多模态模型才支持。

一些支持多模态的模型有：
- qwen3.5-plus
- gpt-5-nano
- ...

我们以qwen3.5-plus为例，演示向模型发送图片消息

## 2.1.在线图片
首先，我们演示如何发送一个在线图片给模型，也就是指定模型的url地址。
图片如下：

<img src="https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg" width="500" height="300" alt="图片描述">



In [18]:
from langchain.chat_models import init_chat_model
import os

# 初始化模型
model = init_chat_model(
    model="qwen3.5-plus",  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

In [19]:
# 创建Agent
agent = create_agent(model=model)

In [5]:
# 准备多模态消息
message = HumanMessage([
        {"type": "text", "text": "描述以下这张图片的内容."},
        {"type": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
    ])

In [6]:
stream = agent.stream(
    {"messages": [message]},
    stream_mode="messages"
)
for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


BadRequestError: Error code: 400 - {'error': {'message': 'Failed to deserialize the JSON body into the target type: messages[0]: unknown variant `image_url`, expected `text` at line 1 column 246', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

## 2.2.本地图片数据
有时候用户会上传图片数据，而不是图片的url地址。我们需要将图片数据转换成base64字符串，然后发送给模型。

接下来我们会模拟图片上传、转换的过程。

首先，我们安装一个上传组件，用于模拟图片上传。


In [ ]:
!uv add ipywidgets

Resolved 150 packages in 2.25s                                       
Installed 3 packages in 38ms.0.16                                ░░░░░░░░░░░░░░░░░░░░ [0/0] Installing wheels...                                 
 + ipywidgets==8.1.8
 + jupyterlab-widgets==3.0.16
 + widgetsnbextension==4.0.15


然后，我们创建一个上传组件，用于模拟图片上传。


In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*', multiple=False)
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [ ]:
print(uploader.value)

({'name': 'fMrWp9r43.jpeg', 'type': 'image/jpeg', 'size': 232405, 'content': <memory at 0x000001E2B6E0EE00>, 'last_modified': datetime.datetime(2026, 3, 2, 8, 54, 44, 188000, tzinfo=datetime.timezone.utc)},)


In [25]:
# 读取图片，转为base64字符串
import base64

# 获取第一个（也是唯一一个）上传的文件
uploaded_file = uploader.value[0]

# 获取其内存视图
content_mv = uploaded_file["content"]

# 转换内存视图->字节
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [26]:
# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片中的城市"}
])

for chunk, metadata in agent.stream(
    {"messages": [multimodal_question]},
    stream_mode="messages"
):
    print(chunk.content, end="", flush=True)

这张图片展示了一个宏大且充满未来感的**科幻城市**，看起来像是位于外星球殖民地或者地球遥远的未来。以下是关于这座城市的详细描述：

**1. 壮丽的天空与背景**
*   **巨大的天体**：最引人注目的是右上角悬挂着一颗巨大的星球（可能是卫星或邻近的行星）。它占据了很大一部分天空，表面闪烁着点点光芒，仿佛上面也有文明存在，或者仅仅是反射了恒星的光辉。
*   **璀璨星空**：背景是深邃的宇宙，繁星密布，甚至能看到淡淡的银河。这暗示着这座城市的大气层可能非常稀薄，或者空气极度纯净，能见度极高。
*   **雪山环绕**：城市远处被连绵起伏的雪山环绕，给这座高科技城市增添了一丝苍凉和壮阔的自然背景。

**2. 独特的建筑风格**
*   **左侧地标塔**：在画面左侧，矗立着一座巨大的尖塔状建筑。它呈圆锥形，通体由网格状的灯光覆盖，像是一座巨大的信号塔、能量塔或者是城市的中心摩天楼。它的底部是一个宽阔的圆盘基座，层层叠叠，极具气势。
*   **右侧前景建筑**：右下角是一个巨大的圆形建筑，看起来像是一个巨大的飞碟或圆顶竞技场。它的边缘有明亮的环形灯带，内部结构复杂，充满了金属质感和机械美学。
*   **城市肌理**：整个城市由无数低矮但密集的建筑组成，大多采用流线型设计，屋顶多为圆顶或扁平状，充满了“太空时代”的审美。

**3. 繁华的城市生活与交通**
*   **灯火通明**：尽管是夜晚（或处于背阴面），城市却灯火辉煌。无数暖黄色的灯光从建筑物中透出，与冷色调的金属外壳形成对比，显示出这里人口稠密，充满活力。
*   **水陆交通**：画面左下方似乎有水域（河流或湖泊），倒映着城市的灯光。岸边停靠着几艘细长的、类似飞船或潜水艇的交通工具，暗示这里可能有发达的水路或低空飞行交通网络。
*   **道路系统**：可以看到宽阔的道路连接着各个区域，地面上似乎还有车辆在行驶。

**总结**
这就好像是一部太空歌剧电影中的场景。这座城市融合了**高科技（High-Tech）**与**自然奇观**，给人一种既冰冷又温暖、既孤独又繁华的视觉冲击。它可能是一个人类在星际旅行中建立的前哨站，或者是一个高度发达的外星文明都市。